In [1]:
import torch
from datasets import load_dataset

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    GenerationConfig,
    pipeline
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer



In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [3]:
config = AutoConfig.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
print("Model Type:", config.model_type)
print("Hidden Size:", config.hidden_size)
print("Vocab Size:", config.vocab_size)
print("Num Layers:", config.num_hidden_layers)

Model Type: llama
Hidden Size: 2048
Vocab Size: 32000
Num Layers: 22


In [5]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

In [6]:
model = AutoModelForCausalLM.from_pretrained(

    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [7]:
model.config.use_cache = False

In [8]:
model = prepare_model_for_kbit_training(model)

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "Right"

In [10]:
peft_config = LoraConfig(

    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [11]:
model = get_peft_model(
    model,
    peft_config
)

In [12]:
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [13]:

dataset = load_dataset("mlabonne/guanaco-llama2-1k",split="train[:200]")

In [14]:
print("\nDataset Size:", len(dataset))


Dataset Size: 200


In [15]:
def clean_dataset(example):

    return {"text": example["text"].strip()}

dataset = dataset.map(clean_dataset)

In [16]:
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]


In [17]:
sample = train_dataset[0]["text"]

tokens = tokenizer(
    sample,
    return_tensors="pt"
)

print("\nTokenizer Output Keys:")
print(tokens.keys())



Tokenizer Output Keys:
KeysView({'input_ids': tensor([[    1,     1, 29961, 25580, 29962, 18613,  9832,   267,   712,  1232,
         17541,  1008,  5060,   690, 14842,   273,  6493,   912,  9343,   425,
         27907,   427,   560,  2331,   912, 29973,   518, 29914, 25580, 29962,
         17295, 13856,  2101,  1512, 23116, 29892,  3737,  2090,  1290,   831,
         24719, 12401,   279, 29717, 13413,  4244,   343,  2362,  1114,   427,
           540, 18688,   752, 13716,  1849, 29889,   379,  5427,   560, 14341,
           694, 14842, 28854,  6500, 26534,  8631,  5326,   712,  1261, 16139,
          1267,   712,  1232, 17541,  1008,  5060,   690, 14842,   273,  6493,
           912,   425,   323, 11617,   427,   560,  2331,   912, 29889,   319,
         11246,   447,  2299,  1941, 24312,  1871,   267,   316,  1029,   391,
           314, 14948,   316, 15397,  6994,   343, 11929, 18371, 29980,  1527,
           359,   297,  4548,   506,  1849, 29892,   425, 26960,   316, 21010,
     

In [18]:
with torch.no_grad():

    outputs = model(

        input_ids=tokens["input_ids"].to(model.device),

        attention_mask=tokens["attention_mask"].to(model.device)
    )

print("\nOutput Type:")
print(type(outputs))

print("\nLogits Shape:")
print(outputs.logits.shape)


Output Type:
<class 'transformers.modeling_outputs.CausalLMOutputWithPast'>

Logits Shape:
torch.Size([1, 191, 32000])


In [19]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=1,

    per_device_train_batch_size=1,

    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    weight_decay=0.001,

    warmup_ratio=0.03,

    max_grad_norm=0.3,

    bf16=True,

    logging_steps=10,

    save_strategy="steps",

    save_steps=50,

    eval_strategy="steps",

    eval_steps=50,

    save_total_limit=2,

    report_to="none"
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [20]:
trainer = SFTTrainer(

    model=model,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    processing_class=tokenizer,

    args=training_args
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)


In [21]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
23,1.672298,2.001998,1.969425,0.575457,75605.000000


TrainOutput(global_step=23, training_loss=1.6585671590722126, metrics={'train_runtime': 295.4529, 'train_samples_per_second': 0.609, 'train_steps_per_second': 0.078, 'total_flos': 471329728081920.0, 'train_loss': 1.6585671590722126, 'epoch': 1.0})

In [22]:
adapter_path = "./tinyllama_lora_adapter"

trainer.model.save_pretrained(
    adapter_path
)

tokenizer.save_pretrained(
    adapter_path
)


('./tinyllama_lora_adapter/tokenizer_config.json',
 './tinyllama_lora_adapter/chat_template.jinja',
 './tinyllama_lora_adapter/tokenizer.json')

In [23]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [24]:
adapter_path = '/content/results/checkpoint-23/'

model = PeftModel.from_pretrained(

    base_model,

    adapter_path
)

In [25]:
import torchao

print(torchao.__version__)

0.17.0


In [26]:
!pip install -U torchao

In [26]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained(
    "./merged_model"
)

tokenizer.save_pretrained(
    "./merged_model"
)

print("\nMerged Model Saved")


In [28]:
generation_config = GenerationConfig(

    max_new_tokens=128,

    temperature=0.7,

    top_p=0.9,

    top_k=50,

    repetition_penalty=1.1,

    do_sample=True
)

print("\nGeneration Config Created")


Generation Config Created


In [30]:
merged_model = model.merge_and_unload()

prompt = "Explain neural networks."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(merged_model.device)

generated_tokens = merged_model.generate(

    **inputs,

    generation_config=generation_config
)

response = tokenizer.decode(

    generated_tokens[0],

    skip_special_tokens=True
)

print("Inference")
print(response)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inference
Explain neural networks.


In [31]:
generator = pipeline(

    "text-generation",

    model=merged_model,

    tokenizer=tokenizer
)

result = generator(

    "Explain machine learning.",

    generation_config=generation_config
)

print("\nPipeline Output")
print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Pipeline Output
Explain machine learning.
